# 01 — Dataset Audit & Project-Fit Check

In [1]:
import os
import yaml

In [2]:
PATH_MOSQUITOFUSION = "data/raw/MosquitoFusion Dataset"
PATH_VISTEXT_ROOT    = "data/raw/VisText-Mosquito A Multimodal Dataset for Mosquito"
PATH_VISTEXT_DETECT  = os.path.join(PATH_VISTEXT_ROOT, "Breeding Place Detection")

def read_data_yaml(root):
    """Reads a Roboflow-style data.yaml and returns the class name list."""
    yaml_path = os.path.join(root, "data.yaml")
    if not os.path.isfile(yaml_path):
        print(f"[!] No data.yaml found in {root}")
        return None
    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    return data.get("names")

def detect_splits(root):
    """Finds split folders (train/valid/test, or train/val/test, etc.) that contain images+labels."""
    found = {}
    for name in os.listdir(root):
        split_dir = os.path.join(root, name)
        if not os.path.isdir(split_dir):
            continue
        img_dir = os.path.join(split_dir, "images")
        lbl_dir = os.path.join(split_dir, "labels")
        if os.path.isdir(img_dir) and os.path.isdir(lbl_dir):
            found[name] = {"images": img_dir, "labels": lbl_dir}
    return found


## 1. Audit MosquitoFusion (auto-detects train/valid/test)

In [3]:
mf_splits = detect_splits(PATH_MOSQUITOFUSION)
mf_classes = read_data_yaml(PATH_MOSQUITOFUSION)
print("MosquitoFusion classes:", mf_classes)
for split, dirs in mf_splits.items():
    n_img = len(os.listdir(dirs["images"]))
    n_lbl = len(os.listdir(dirs["labels"]))
    print(f"{split:8s} -> images: {n_img:5d}   labels: {n_lbl:5d}")
if not mf_splits:
    print("[!] No split folders with images/labels found under", PATH_MOSQUITOFUSION,
          "- check the folder was unzipped correctly.")

MosquitoFusion classes: ['Breeding Place', 'Mosquito', 'Mosquito Swarm']
test     -> images:    51   labels:    51
train    -> images:  1053   labels:  1053
valid    -> images:   100   labels:   100


## 2. Audit VisText-Mosquito — Breeding Place Detection subset

In [4]:
vt_splits = detect_splits(PATH_VISTEXT_DETECT)
vt_classes = read_data_yaml(PATH_VISTEXT_DETECT)
print("VisText-Mosquito (Breeding Place Detection) classes:", vt_classes)
for split, dirs in vt_splits.items():
    n_img = len(os.listdir(dirs["images"]))
    n_lbl = len(os.listdir(dirs["labels"]))
    print(f"{split:8s} -> images: {n_img:5d}   labels: {n_lbl:5d}")
if not vt_splits:
    print("[!] No split folders found directly under 'Breeding Place Detection'.")
    print("    Open that folder and check its actual layout - it may itself contain")
    print("    train/valid/test subfolders, or a single images/ + labels/ pair.")
    print("    Adjust PATH_VISTEXT_DETECT above once you've confirmed the real layout.")

VisText-Mosquito (Breeding Place Detection) classes: ['Bottle', 'Coconut-Exocarp', 'Drain-Inlet', 'Tire', 'Vase']
test     -> images:   183   labels:   183
train    -> images:  3871   labels:  3871
valid    -> images:   371   labels:   371
